# Create Pad Foundations - Revit 2025

Notebook template for the `automation-api-footings` APS Design Automation workflow.

Inputs:
- `input.rvt`
- `pad_foundations.json`

Output:
- `result.rvt`


In [ ]:
import os
import json
import logging
import uuid
from pathlib import Path
from dotenv import load_dotenv

from aps_automation_sdk.classes import (
    Activity,
    ActivityInputParameter,
    ActivityOutputParameter,
    AppBundle,
    WorkItem,
)

from aps_automation_sdk.utils import (
    delete_activity,
    delete_appbundle,
    get_token,
    set_nickname,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


In [ ]:
load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID", "")
CLIENT_SECRET = os.getenv("CLIENT_SECRET", "")

token = get_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
nickname = set_nickname(token, "myUniqueNickNameHere")

print(f"Authentication successful. Nickname: {nickname}")


In [ ]:
YEAR = 2025

app_bundle_name = f"PadFoundationsDA{YEAR}"
activity_name = f"PadFoundationsActivity{YEAR}"
alias = "dev"
bucket_key = uuid.uuid4().hex

zip_path = Path.cwd() / "2025 - Net8" / "files" / "PadFoundationsDA.bundle.zip"
input_rvt_path = Path.cwd() / "2025 - Net8" / "files" / "StructuralTemplate.rvt"
input_json_path = Path.cwd() / "2025 - Net8" / "files" / "pad_foundations.sample.json"
output_rvt_path = Path.cwd() / "2025 - Net8" / "files" / "output" / "result.rvt"

appbundle_full_alias = f"{nickname}.{app_bundle_name}+{alias}"
activity_full_alias = f"{nickname}.{activity_name}+{alias}"

print(f"App Bundle: {appbundle_full_alias}")
print(f"Activity:   {activity_full_alias}")
print(f"Bundle ZIP: {zip_path}")
print(f"Input RVT:  {input_rvt_path}")
print(f"Input JSON: {input_json_path}")


In [ ]:
bundle = AppBundle(
    appBundleId=app_bundle_name,
    engine=f"Autodesk.Revit+{YEAR}",
    alias=alias,
    zip_path=str(zip_path),
    description=f"Create pad foundations for Revit {YEAR}",
)

bundle.deploy(token)
print("App bundle deployed successfully!")


In [ ]:
input_revit = ActivityInputParameter(
    name="inputModel",
    localName="input.rvt",
    verb="get",
    description="Input Revit model",
    required=True,
    is_engine_input=True,
    bucketKey=bucket_key,
    objectKey="input.rvt",
)

input_json = ActivityInputParameter(
    name="footingPayload",
    localName="pad_foundations.json",
    verb="get",
    description="Pad foundations JSON payload",
    required=True,
    bucketKey=bucket_key,
    objectKey="pad_foundations.json",
)

output_file = ActivityOutputParameter(
    name="resultModel",
    localName="result.rvt",
    verb="put",
    description="Output Revit model",
    bucketKey=bucket_key,
    objectKey="result.rvt",
)

print("Activity parameters defined successfully!")


In [ ]:
activity = Activity(
    id=activity_name,
    parameters=[input_revit, input_json, output_file],
    engine=f"Autodesk.Revit+{YEAR}",
    appbundle_full_name=appbundle_full_alias,
    description=f"Create pad foundations in Revit {YEAR} from JSON",
    alias=alias,
)

activity.set_revit_command_line()
activity.deploy(token=token)
print("Activity deployed successfully!")


In [ ]:
input_revit.upload_file_to_oss(file_path=str(input_rvt_path), token=token)
input_json.upload_file_to_oss(file_path=str(input_json_path), token=token)
print(f"Uploaded input files: {input_rvt_path.name}, {input_json_path.name}")


In [ ]:
work_item = WorkItem(
    parameters=[input_revit, input_json, output_file],
    activity_full_alias=activity_full_alias,
)

print("Work item created. Starting execution...")


In [ ]:
status_resp = work_item.execute(token=token, max_wait=900, interval=10)
last_status = status_resp.get("status", "")

print(f"Work item completed with status: {last_status}")
print(json.dumps(status_resp, indent=2))


In [ ]:
if last_status == "success":
    output_rvt_path.parent.mkdir(parents=True, exist_ok=True)
    output_file.download_to(output_path=str(output_rvt_path), token=token)
    print(f"Download successful: {output_rvt_path}")
else:
    print(f"Work item failed or did not complete successfully. Status: {last_status}")


In [ ]:
try:
    delete_activity(activityId=activity_name, token=token)
    print(f"Activity deleted: {activity_full_alias}")
except Exception as exc:
    print(f"Failed to delete activity: {exc}")

try:
    delete_appbundle(appbundleId=app_bundle_name, token=token)
    print(f"App bundle deleted: {app_bundle_name}")
except Exception as exc:
    print(f"Failed to delete app bundle: {exc}")
